# QAT · Vision SFT · LaTeX OCR

Try `mlx-community/Qwen3.5-2B-4bit` with weight-only affine QAT through the
shared training API. The loader reads the packed checkpoint; preparation
**dequantizes it to floating-point master weights**, then simulates 4-bit
quantization on eligible standard Linear layers with straight-through gradients.
Embeddings and unsupported layers stay floating point. This does not recover
the original pre-quantization weights.

This is full fine-tuning, including the vision tower, with full-training memory
and optimizer costs. Text-only batches provide no vision inputs. Start with
these four-step settings, then increase sample limits and iterations.
Checkpoints contain float master weights, not packed 4-bit inference weights.
Resume with the same QAT settings; export/quantize separately for inference.

## Setup

Use an Apple Silicon Mac. Install into the notebook kernel, then restart it:

```sh
python -m pip install -e "/absolute/path/to/mlx-vlm[train]" jupyterlab ipykernel pandas
```

Run cells in order. Model and dataset downloads are cached after first use.


## Setup

Use an Apple Silicon Mac and the Python environment selected by your notebook kernel.
Install once from your checkout, then restart the kernel:

```sh
python -m pip install -e "/absolute/path/to/mlx-vlm[train]" jupyterlab ipykernel
```

Run the cells in order. The first load downloads the model and dataset.


In [ ]:
from pathlib import Path

import mlx.core as mx
import mlx.optimizers as optim
import pandas as pd
from datasets import load_dataset

from mlx_vlm import load
from mlx_vlm.prompt_utils import apply_chat_template
from mlx_vlm.trainer import Trainer, VLMTrainingArgs, prepare_model_for_training
from mlx_vlm.trainer.datasets import CacheDataset
from mlx_vlm.trainer.vlm.sft.dataset import VisionDataset

## 1. Settings

Four microsteps provide a short smoke run. Increase iters and sample limits for
longer experiments. QAT uses full fine-tuning; keep TRAIN_TYPE="full" and QAT=True.
QAT_BITS accepts 4 or 8, and QAT_GROUP_SIZE accepts 32, 64, or 128.
Paths are relative to the notebook kernel's working directory.


In [ ]:
MODEL_ID = "mlx-community/Qwen3.5-2B-4bit"
DATASET_ID = "mlx-community/LaTeX_OCR"
VALIDATION_FRACTION = 0.02
MAX_TRAIN_SAMPLES = 32
MAX_EVAL_SAMPLES = 4
OCR_PROMPT = (
    "Transcribe the mathematical expression in this image into LaTeX. "
    "Return only the LaTeX expression, without explanations or Markdown fences."
)
OUTPUT_DIR = Path("checkpoints/vlm/qat_sft_vision")
SEED = 42

# Full QAT
TRAIN_TYPE = "full"
QAT = True
QAT_BITS = 4
QAT_GROUP_SIZE = 64
CHECKPOINT_PATH = None

# Training
args = VLMTrainingArgs(
    iters=4,
    batch_size=1,
    gradient_accumulation_steps=4,
    max_seq_length=512,
    train_on_completions=True,
    cache_size=8,
    grad_checkpoint=True,
    compile=False,
    steps_per_report=1,
    steps_per_eval=4,
    steps_per_save=4,
    adapter_file=str(OUTPUT_DIR / "weights.safetensors"),
)
mx.random.seed(SEED)
args.validate()


## 2. Data

Hold out 2% of the training split for validation; keep the original test split for
final evaluation. `.map()` converts `text` into user/assistant messages and retains
the raw `image` column.


In [ ]:
raw_dataset = load_dataset(DATASET_ID)
required_columns = {"image", "text"}
if not required_columns.issubset(raw_dataset["train"].column_names):
    raise ValueError(
        f"Expected image/text columns, got {raw_dataset['train'].column_names}."
    )

partitions = raw_dataset["train"].train_test_split(
    test_size=VALIDATION_FRACTION,
    seed=SEED,
)


def limit_rows(rows, maximum):
    if maximum is None:
        return rows
    if maximum < 1:
        raise ValueError("Sample limits must be positive or None.")
    return rows.select(range(min(maximum, len(rows))))


def latex_to_messages(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Every OCR row must contain a non-empty LaTeX text string.")
    return {
        "messages": [
            {"role": "user", "content": OCR_PROMPT},
            {"role": "assistant", "content": text},
        ],
    }


train_rows = limit_rows(partitions["train"], MAX_TRAIN_SAMPLES).map(
    latex_to_messages,
    input_columns=["text"],
    remove_columns=["text"],
    desc="Map training LaTeX into chat messages",
)
eval_rows = limit_rows(partitions["test"], MAX_EVAL_SAMPLES).map(
    latex_to_messages,
    input_columns=["text"],
    remove_columns=["text"],
    desc="Map validation LaTeX into chat messages",
)

assert len(train_rows) >= args.batch_size, "Reduce batch_size or add training rows."
assert len(eval_rows) >= args.batch_size, "Validation needs a complete batch."
print({"train_rows": len(train_rows), "validation_rows": len(eval_rows)})
print(train_rows.features)

## 3. Model

Preview the model's chat template with one image and the assistant target included.
Image decoding and token expansion happen during dataset preprocessing.


In [ ]:
model, processor = load(MODEL_ID)
sample = train_rows[0]
formatted_sample = apply_chat_template(
    processor,
    model.config,
    sample["messages"],
    num_images=1,
    add_generation_prompt=False,
)
print({"image_size": sample["image"].size})
print(formatted_sample[:2000])

## Prepare full QAT

Load the requested packed model directly. Preparation dequantizes packed weights,
then replaces compatible standard Linear layers with local QAT layers. No global
class patches or separate training loop are needed. Set CHECKPOINT_PATH to a full
checkpoint to resume with a fresh optimizer and the same QAT settings.


In [ ]:
model = prepare_model_for_training(
    model,
    train_type=TRAIN_TYPE,
    qat=QAT,
    quantization_bits=QAT_BITS,
    quantization_group_size=QAT_GROUP_SIZE,
    checkpoint_path=CHECKPOINT_PATH,
)


## 4. Train

VisionDataset applies the chat template, processes the raw image, and masks the prompt.
Only the LaTeX response contributes to the loss.


In [ ]:
train_vision_dataset = VisionDataset(
    train_rows,
    model.config,
    processor,
    config=args,
)
eval_vision_dataset = VisionDataset(
    eval_rows,
    model.config,
    processor,
    config=args,
)

optimizer = optim.AdamW(
    learning_rate=1e-5,
    weight_decay=0.0,
)

trainer = Trainer(
    model,
    args,
    optimizer=optimizer,
    task="vlm",
    algorithm="sft",
    processing_class=processor,
    train_dataset=CacheDataset(
        train_vision_dataset, max_size=args.cache_size, process=False
    ),
    eval_dataset=CacheDataset(
        eval_vision_dataset, max_size=args.cache_size, process=False
    ),
    prepared=True,
    seed=SEED,
)

In [ ]:
metrics = trainer.train()

### Metrics

The table shows the last five training reports. All metrics remain in `metrics`
and `trainer.log_history`, including loss diagnostics, timing, and memory.
`trained_tokens` counts supervised targets; `total_vision_tokens` counts expanded
image/video input positions. Processing token/s includes forward, backward, and updates.


In [ ]:
report_history = pd.DataFrame(trainer.log_history)
columns = {
    "iteration": "step",
    "train_loss": "loss",
    "learning_rate": "lr",
    "trained_tokens": "trained tokens",
    "total_vision_tokens": "vision tokens",
    "processed_tokens_per_second": "processed tok/s",
    "peak_memory": "peak GB",
}
recent_reports = (
    report_history.dropna(subset=["train_loss"])
    .tail(5)
    .reindex(columns=columns)
    .rename(columns=columns)
    .set_index("step")
)
recent_reports.round({"loss": 4, "processed tok/s": 1, "peak GB": 2})

## 5. Evaluate

Check response loss on held-out images. To assess OCR quality, also compare generated
LaTeX with the targets on the original test split.


In [ ]:
assert mx.isfinite(mx.array(metrics["train_loss"])).item(), "Non-finite training loss."
validation = trainer.evaluate() if eval_rows is not None else {}
pd.DataFrame([validation]).filter(
    items=["val_loss", "val_token_accuracy", "val_perplexity", "val_num_vision_tokens"]
).round(4)

## 6. Save

Keep `weights.safetensors` and `adapter_config.json` together. The loop also saves
the final checkpoint automatically.


In [ ]:
trainer.save_adapter()
print(Path(args.adapter_file).resolve())

### Reload with the same QAT settings

```python
model, processor = load(MODEL_ID)
model = prepare_model_for_training(
    model, train_type="full", qat=True,
    quantization_bits=QAT_BITS, quantization_group_size=QAT_GROUP_SIZE,
    checkpoint_path=args.adapter_file,
)
model.eval()
```

Evaluation retains fake quantization. The full checkpoint sidecar does not record
QAT settings, so keep this notebook's settings with the weights.
See the [training guide](../../docs/training.md).
